# BER full pipeline — ONE self-contained notebook (stages A-H)

Every function lives in this notebook (inlined from src via AST build — no external
.py, no subprocess). Only external reads: the competition dataset/ TSVs.

| Cell | Stage | Does | ~Time (full) |
|---|---|---|---|
| 4 | A prep | normalize 6 TSVs -> work/*.parquet + slim id lookups | 30-40 min |
| 5 | B sweep | recall@K table (K x proxy x caps) on val split -> pick TOPK/PROXY | 40-60 min |
| 6 | C cands | train candidates with TOPK/PROXY | 40-80 min |
| 7 | D feats | train features + labels + neg downsample | 1-1.5 h |
| 8 | E gate | adversarial identical-name check (PASS required) | 2 min |
| 9 | F train | LightGBM + ES + F_0.5 sweep -> model artifacts | 1-2 h |
| 10 | G infer | test inference + inline validator (PASS required) | ~1 h |
| 11 | H zip | kaggle_return.zip | 1 min |

CONFIG cell: SAMPLE_N>0 runs a small-slice end-to-end test; set SAMPLE_N=0 for the
real run. Stage B selects caps/proxy/candidate cutoff automatically; running the notebook
top-to-bottom completes stages A-H in one session. Same-session reuse requires a matching
completion manifest and intact outputs.

In [ ]:
# S0a — setup: deps (install missing only, never pin over the image), paths, globals
import importlib, subprocess, sys
for mod, package in (("pandas", "pandas"), ("numpy", "numpy"),
                     ("pyarrow", "pyarrow"), ("lightgbm", "lightgbm"),
                     ("sklearn", "scikit-learn")):
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
import glob, os, re, json, time, shutil, zipfile, hashlib, argparse, ast
import numpy as np
import pandas as pd
import lightgbm as lgb

# mount detection: handles dataset/train/ + dataset/test/ layout AND flat layout
def find_data_dir():
    if os.environ.get("BER_DATA"):
        return os.path.abspath(os.environ["BER_DATA"])
    for m in glob.glob("/kaggle/input/*"):
        for base, dirs, files in os.walk(m):
            if "train_source1.tsv" in files:
                if "test_source1.tsv" in files:
                    return base                       # flat layout
                parent = os.path.dirname(base.rstrip("/"))
                if os.path.basename(base) == "train" and \
                        os.path.exists(os.path.join(parent, "test", "test_source1.tsv")):
                    return parent                      # split train/ + test/ layout
    raise FileNotFoundError("train/test source TSVs not found under /kaggle/input — attach the payload dataset")

DATA_DIR = find_data_dir()
BASE = DATA_DIR                       # prep's name for it
if os.environ.get("BER_WORK"):
    WORK = os.path.abspath(os.environ["BER_WORK"])
elif os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working/work"
else:
    WORK = os.path.abspath("work_nb")
OUTPUT_DIR = os.path.join(os.path.dirname(WORK), "output") if os.environ.get("BER_WORK") \
    else "/kaggle/working/output"
GT = os.path.join(WORK, "ground_truth.tsv")
# ground truth: support nested and flat layouts, and refresh if the attached data changes.
gt_src = os.path.join(DATA_DIR, "train", "train_ground_truth.tsv")
if not os.path.exists(gt_src):
    gt_src = os.path.join(DATA_DIR, "train_ground_truth.tsv")
if not os.path.isfile(gt_src):
    raise FileNotFoundError("train_ground_truth.tsv not found in flat or split dataset layout")
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
if not os.path.exists(GT) or file_sha256(GT) != file_sha256(gt_src):
    os.makedirs(WORK, exist_ok=True)
    shutil.copy2(gt_src, GT)
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("WORK:", WORK, "| OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# S0b — CONFIG (the only cell you edit between stages)
SAMPLE_N = 0           # >0: small-slice end-to-end test. 0 = FULL
TOPK = 20              # emergency relaunch: largest K that fits the remaining clock
PROXY = "mean"         # 'mean' or 'max' — full-scale Stage B: mean > max at every K
NEG_RATIO = 0.05      # train-side negative keep-rate; val rows + positives always kept
CAPS = {"token_cap": 1000, "addr_cap": 2000, "sn_cap": 2000, "fn_cap": 20000, "c3_cap": 2000}
MAX_CANDIDATES = 2000
PAIR_BUDGET = 100_000
KS = [10, 20, 30, 50, 100, 200] if SAMPLE_N == 0 else [5, 10, 20, 50]
_SETTINGS = [(1000, 2000, 2000, 20000, 2000), (200, 300, 500, 5000, 1000)] if SAMPLE_N == 0 else \
            [(50, 100, 100, 1000, 200), (20, 30, 50, 300, 100)]
FEATURES = ["name_jaccard_raw", "name_jaccard_normalized", "name_jaccard_suffix_stripped",
            "name_cos", "addr_cos", "addr_jaccard", "token_overlap_count",
            "street_number_match", "missing_address_flag",
            "len_delta_name", "len_delta_addr"]
print("CONFIG: SAMPLE_N", SAMPLE_N, "| TOPK", TOPK, "| PROXY", PROXY, "| NEG_RATIO", NEG_RATIO)

In [ ]:
# S1a — normalization functions (inlined normalize.py)
import unicodedata
import re

import unicodedata

_LETTERS = {'A': 'a', 'AA': 'aa', 'I': 'i', 'II': 'ii', 'U': 'u', 'UU': 'uu', 'VOCALIC R': 'ri', 'VOCALIC RR': 'ri', 'VOCALIC L': 'li', 'VOCALIC LL': 'li', 'CANDRA E': 'e', 'SHORT E': 'e', 'E': 'e', 'EE': 'e', 'AI': 'ai', 'CANDRA O': 'o', 'SHORT O': 'o', 'O': 'o', 'OO': 'o', 'AU': 'au', 'SHORT A': 'a', 'CANDRA A': 'a', 'OE': 'o', 'OOE': 'o', 'AW': 'o', 'UE': 'u', 'UUE': 'u', 'KA': 'k', 'KHA': 'kh', 'GA': 'g', 'GHA': 'gh', 'NGA': 'n', 'CA': 'ch', 'CHA': 'ch', 'JA': 'j', 'JHA': 'jh', 'NYA': 'n', 'TTA': 't', 'TTHA': 'th', 'DDA': 'd', 'DDHA': 'dh', 'NNA': 'n', 'NNNA': 'n', 'TA': 't', 'THA': 'th', 'DA': 'd', 'DHA': 'dh', 'NA': 'n', 'PA': 'p', 'PHA': 'ph', 'BA': 'b', 'BHA': 'bh', 'MA': 'm', 'YA': 'y', 'RA': 'r', 'RRA': 'r', 'LA': 'l', 'LLA': 'l', 'LLLA': 'l', 'VA': 'v', 'SHA': 'sh', 'SSA': 'sh', 'SA': 's', 'HA': 'h', 'QA': 'q', 'KHHA': 'kh', 'GHHA': 'g', 'ZA': 'z', 'DDDHA': 'r', 'RHA': 'r', 'FA': 'f', 'YYA': 'y', 'GGA': 'g', 'JJA': 'j', 'DDDA': 'd', 'BBA': 'b', 'ZHA': 'l', 'MARWARI DDA': 'd', 'HEAVY YA': 'y', 'GLOTTAL STOP': '', 'AVAGRAHA': '', 'OM': ''}

_SIGNS = {'ANUSVARA': 'n', 'CANDRABINDU': '', 'INVERTED CANDRABINDU': '', 'VISARGA': 'h', 'NUKTA': '', 'VIRAMA': '', 'VOWEL SIGN AA': 'aa', 'VOWEL SIGN I': 'i', 'VOWEL SIGN II': 'ii', 'VOWEL SIGN U': 'u', 'VOWEL SIGN UU': 'uu', 'VOWEL SIGN VOCALIC R': 'ri', 'VOWEL SIGN VOCALIC RR': 'ri', 'VOWEL SIGN VOCALIC L': 'li', 'VOWEL SIGN VOCALIC LL': 'li', 'VOWEL SIGN E': 'e', 'VOWEL SIGN SHORT E': 'e', 'VOWEL SIGN CANDRA E': 'e', 'VOWEL SIGN AI': 'ai', 'VOWEL SIGN O': 'o', 'VOWEL SIGN SHORT O': 'o', 'VOWEL SIGN CANDRA O': 'o', 'VOWEL SIGN AU': 'au', 'VOWEL SIGN PRISHTHAMATRA E': 'e', 'VOWEL SIGN AW': 'o', 'VOWEL SIGN OE': 'o', 'VOWEL SIGN OOE': 'o', 'VOWEL SIGN UE': 'u', 'VOWEL SIGN UUE': 'u', 'VOWEL SIGN CANDRA LONG E': 'e', 'AU LENGTH MARK': 'au'}

_TR = {}

for _cp in range(2304, 3456):
    _n = unicodedata.name(chr(_cp), '')
    for _prefix, _tab in (('DEVANAGARI ', None), ('GUJARATI ', None), ('TAMIL ', None), ('TELUGU ', None), ('KANNADA ', None), ('MALAYALAM ', None), ('ORIYA ', None), ('GURMUKHI ', None), ('BENGALI ', None)):
        if _n.startswith(_prefix):
            _rest = _n[len(_prefix):]
            if _rest.startswith('LETTER '):
                _t = _LETTERS.get(_rest[7:])
            elif _rest.startswith('VOWEL SIGN '):
                _t = _SIGNS.get(_rest, _SIGNS.get('VOWEL SIGN ' + _rest[11:]))
            elif _rest.startswith('SIGN '):
                _t = _SIGNS.get(_rest[5:])
            elif _rest.startswith('DIGIT '):
                _t = {'ZERO': '0', 'ONE': '1', 'TWO': '2', 'THREE': '3', 'FOUR': '4', 'FIVE': '5', 'SIX': '6', 'SEVEN': '7', 'EIGHT': '8', 'NINE': '9'}.get(_rest[6:])
            else:
                _t = ' '
            if _t is not None:
                _TR[_cp] = _t
            break

_TRANSLATE_TABLE = {**{cp: ' ' for cp in range(2304, 3456)}, **_TR, 8204: None, 8205: None}

def transliterate(s):
    if s.isascii():
        return s
    return s.translate(_TRANSLATE_TABLE)

_WS = re.compile('\\s+')

_NONALNUM = re.compile('[^a-z0-9]+')

def norm_text(s):
    """lowercase + transliterate + fold Latin diacritics + punct->space + collapse.

    Order matters: NFKC-compose, transliterate Indic->ASCII (needs composed input),
    THEN NFKD-decompose + strip combining marks so accented Latin folds to base
    letters (café->cafe) instead of being deleted by the alnum filter. France is
    untestable against train labels, so this path is guarded by a self-check below.
    """
    s = unicodedata.normalize('NFKC', s or '')
    s = transliterate(s).lower()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join((c for c in s if unicodedata.category(c) != 'Mn'))
    s = _NONALNUM.sub(' ', s)
    return _WS.sub(' ', s).strip()

_SUFFIXES = {'inc', 'incorporated', 'llc', 'llp', 'pvt', 'ltd', 'limited', 'corp', 'corporation', 'company', 'co', 'gmbh', 'plc', 'trust', 'private'}

_PUNCTY = set('.,[](){}\'"')

def strip_suffixes(name_norm):
    """Strip legal suffix tokens from both ends of an already-normalized name.

    Tokens are compared with wrapper punctuation (brackets/parens/quotes) removed,
    so "[LLP]" and "(Inc.)" strip like their bare forms.
    """
    toks = name_norm.split()
    while toks and (toks[0].strip('.,[](){}\'"') in _SUFFIXES or all((c in _PUNCTY for c in toks[0]))):
        toks.pop(0)
    while toks and (toks[-1].strip('.,[](){}\'"') in _SUFFIXES or all((c in _PUNCTY for c in toks[-1]))):
        toks.pop()
    return ' '.join(toks)

def norm_name(s):
    """Return (normalized, suffix-stripped) name strings."""
    n = norm_text(s)
    return (n, strip_suffixes(n))

def addr_tokens(s):
    """Loose address tokenization: no order/component assumptions."""
    return norm_text(s).split()

def street_number(addr):
    """Leading house/street number of a normalized address, '' if none."""
    for t in addr_tokens(addr):
        if t[0].isdigit():
            return t
        return ''
    return ''

def _cos_corpus(frame, primary, fallback=None):
    """Text corpus for one TF-IDF fit/transform: `primary` column when present,
    else `fallback` (minimal frames carry name_s but not the derived name_t),
    else all-blank (degenerate corpus -> zero vectors). Pool and query frames
    resolve the same way, so both sides share one vocabulary convention."""
    for col in (primary, fallback):
        if col in frame.columns:
            return frame[col].fillna('').values
    return [''] * len(frame)

def _cos_fit(corpus):
    """Fit the TF-IDF vectorizer; refit without the df floor when a tiny corpus
    prunes the min_df=2 vocabulary empty, and give up (None) only when the
    corpus holds no tokens at all."""
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer

    def vec(min_df):
        return TfidfVectorizer(analyzer='word', token_pattern='(?u)\\S+', min_df=min_df, sublinear_tf=True, dtype=np.float32)
    try:
        v = vec(2)
        return (v, v.fit_transform(corpus))
    except ValueError:
        pass
    try:
        v = vec(1)
        return (v, v.fit_transform(corpus))
    except ValueError:
        return (None, None)

def _cos_fit_frame(frame):
    vn, Xn = _cos_fit(_cos_corpus(frame, 'name_t', 'name_s'))
    va, Xa = _cos_fit(_cos_corpus(frame, 'addr_n'))
    return {'vn': vn, 'va': va, 'Xn': Xn, 'Xa': Xa}

def attach_cosine(pool):
    """Fit TF-IDF on the pool frame once; cache state on pool.attrs."""
    if '_cos' not in pool.attrs:
        pool.attrs['_cos'] = _cos_fit_frame(pool)

def frame_cos_rows(frame, pool):
    """Transform a query frame (s1 frame or row chunk) with the pool vectorizer.

    The cache is tagged with the owning frame's identity: pandas propagates
    attrs through iloc (verified on pandas 3.0), so a chunk sliced from a
    parent that already carries vectors would otherwise reuse the parent-
    sized matrix and score chunk-local positions against the wrong rows."""
    attach_cosine(pool)
    cached = frame.attrs.get('_cos')
    if cached is not None and cached.get('owner') == id(frame):
        return
    st = pool.attrs['_cos']
    xn = st['vn'].transform(_cos_corpus(frame, 'name_t', 'name_s')) if st['vn'] is not None else None
    xa = st['va'].transform(_cos_corpus(frame, 'addr_n')) if st['va'] is not None else None
    frame.attrs['_cos'] = {'owner': id(frame), 'Xn': xn, 'Xa': xa}

def cos_pair_scores(frame, pool, si, ci):
    """Row-wise (name_cos, addr_cos) for frame row si vs pool row ci.
    si/ci are positional integer indices/arrays into frame/pool respectively."""
    import numpy as np
    fs, ps = (frame.attrs['_cos'], pool.attrs['_cos'])
    n = len(np.asarray(si).reshape(-1))
    if fs['Xn'] is not None and ps['Xn'] is not None:
        name = np.asarray(fs['Xn'][si].multiply(ps['Xn'][ci]).sum(axis=1)).ravel()
    else:
        name = np.zeros(n, dtype=np.float64)
    if fs['Xa'] is not None and ps['Xa'] is not None:
        addr = np.asarray(fs['Xa'][si].multiply(ps['Xa'][ci]).sum(axis=1)).ravel()
    else:
        addr = np.zeros(n, dtype=np.float64)
    return (name.astype(np.float64), addr.astype(np.float64))

In [ ]:
# S1b — normalization self-checks (synthetic; France is untestable against train)
assert norm_text("Société Générale") == "societe generale", norm_text("Société Générale")
assert norm_text("Café Crème SARL") == "cafe creme sarl", norm_text("Café Crème SARL")
a = norm_name("Raj Investments LLP")[1]; b = norm_name("ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி")[1]
assert "invest" in b and a == "raj investments", (a, b)
print("S1 self-checks PASS: diacritic fold + Indic transliteration + suffix strip")

In [ ]:
# S2 — Stage A: prep all 6 TSVs + slim id lookups (inlined prep.py)
import json

import os

import shutil

import tempfile

MANIFEST_NAME = '_stage_manifest.json'

def ensure_local_path(path):
    """Normalize an env-derived path and refuse traversal outside its base.

    Stage directories and label paths come from operator-set environment
    variables; this guard keeps a bad value from escaping the intended
    workspace via '..' components."""
    norm = os.path.normpath(os.path.abspath(str(path)))
    if '..' in norm.split(os.sep):
        raise ValueError(f'path escapes its base after normalization: {path}')
    return norm

def files_complete(manifest_path, stage, config, expected_files=None):
    """Validate a manifest for artifacts that live beside or outside it."""
    path = manifest_path
    if not os.path.isfile(path):
        return False
    try:
        with open(path, encoding='utf-8') as f:
            manifest = json.load(f)
        if manifest.get('complete') is not True or manifest.get('stage') != stage or manifest.get('config') != config:
            return False
        files = manifest.get('files', [])
        if not files:
            return False
        for item in files:
            target = item.get('path') or os.path.join(os.path.dirname(path), item['name'])
            if not os.path.isfile(target) or os.path.getsize(target) != item['bytes']:
                return False
        if expected_files is not None and {os.path.abspath(p) for p in expected_files} != {os.path.abspath(item.get('path') or os.path.join(os.path.dirname(path), item['name'])) for item in files}:
            return False
        return True
    except (OSError, ValueError, KeyError, TypeError):
        return False

def is_complete(directory, stage, config):
    """Return true only when config and every recorded output still match."""
    return files_complete(os.path.join(directory, MANIFEST_NAME), stage, config)

def prepare_stage(directory, stage, config):
    """Reuse a verified stage; otherwise clear its own output and start clean."""
    directory = ensure_local_path(directory)
    if is_complete(directory, stage, config):
        return False
    if os.path.isdir(directory):
        shutil.rmtree(directory)
    os.makedirs(directory, exist_ok=True)
    return True

def write_manifest(directory, stage, config, files, total_rows):
    """Atomically mark a stage complete after all outputs have been closed."""
    return write_files_manifest(os.path.join(directory, MANIFEST_NAME), stage, config, files, total_rows)

def write_files_manifest(manifest_path, stage, config, files, total_rows):
    """Atomically record exact output paths, sizes, and row count."""
    directory = os.path.dirname(manifest_path)
    os.makedirs(directory, exist_ok=True)
    items = []
    for path in files:
        if not os.path.isfile(path) or os.path.getsize(path) == 0:
            raise RuntimeError(f'stage output missing or empty: {path}')
        items.append({'name': os.path.basename(path), 'path': os.path.abspath(path), 'bytes': os.path.getsize(path)})
    if not items:
        raise RuntimeError(f'stage {stage} produced no output files')
    manifest = {'stage': stage, 'config': config, 'total_rows': int(total_rows), 'files': items, 'complete': True}
    fd, temp_path = tempfile.mkstemp(prefix='manifest_', suffix='.tmp', dir=directory)
    try:
        with os.fdopen(fd, 'w', encoding='utf-8') as f:
            json.dump(manifest, f, sort_keys=True)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, manifest_path)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)
import sys, os, time, shutil

import pandas as pd

def prep_file(path, out, source_label=None):
    t0 = time.time()
    df = pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)
    nm = df.business_name.map(norm_name)
    df['name_n'] = nm.str[0]
    df['name_s'] = nm.str[1]
    df['name_t'] = df.name_s.map(lambda s: ' '.join(sorted(set(s.split()))))
    df['name_nt'] = df.name_n.map(lambda s: ' '.join(sorted(set(s.split()))))
    df['addr_n'] = df.business_address.map(norm_text)
    df['addr_t'] = df.addr_n.map(lambda s: ' '.join(sorted(set(s.split()))))
    df['stno'] = df.business_address.map(street_number)
    if source_label:
        df['src'] = source_label
    df.to_parquet(out, index=False)
    print(f'{os.path.basename(path)}: {len(df)} rows in {time.time() - t0:.0f}s -> {out}')

def save_slim_ids(split):
    """Slim entity_id lookups (train.py --full / Kaggle reads these, not full parquets)."""
    pd.read_parquet(f'{WORK}/{split}_s1.parquet', columns=['entity_id']).to_parquet(f'{WORK}/{split}_s1_ids.parquet', index=False)
    pd.concat([pd.read_parquet(f'{WORK}/{split}_s2.parquet', columns=['entity_id']), pd.read_parquet(f'{WORK}/{split}_s3.parquet', columns=['entity_id'])], ignore_index=True).to_parquet(f'{WORK}/{split}_pool_ids.parquet', index=False)
    print(f'slim id lookups written for {split}')

prep_cached = {}
for split in ("train", "test"):
    inputs = [os.path.join(DATA_DIR, split, f"{split}_source{i}.tsv") for i in (1, 2, 3)]
    if not os.path.exists(inputs[0]):
        inputs = [os.path.join(DATA_DIR, f"{split}_source{i}.tsv") for i in (1, 2, 3)]
    prep_cfg = {"split": split, "sample_n": SAMPLE_N,
                "ground_truth_sha256": file_sha256(gt_src),
                "source_bytes": [os.path.getsize(p) for p in inputs]}
    prep_manifest = os.path.join(WORK, f"_{split}_prep_manifest.json")
    prep_outputs = [f"{WORK}/{split}_{n}.parquet" for n in ("s1", "s2", "s3")] +                    [f"{WORK}/{split}_s1_ids.parquet", f"{WORK}/{split}_pool_ids.parquet"]
    if files_complete(prep_manifest, f"{split}_prep", prep_cfg, prep_outputs):
        print(f"Stage A {split}: complete and config-matched")
        prep_cached[split] = True
        continue
    prep_cached[split] = False
    for n, label in (("s1", "S1"), ("s2", "S2"), ("s3", "S3")):
        src_tsv = os.path.join(DATA_DIR, split, f"{split}_source{n[1]}.tsv")
        if not os.path.exists(src_tsv):
            src_tsv = os.path.join(DATA_DIR, f"{split}_source{n[1]}.tsv")
        out_pq = f"{WORK}/{split}_{n}.parquet"
        prep_file(src_tsv, out_pq, label)
if SAMPLE_N > 0:
    for split in ("train", "test"):
        if prep_cached.get(split, False):
            continue
        for n in ("s1", "s2", "s3"):
            p = f"{WORK}/{split}_{n}.parquet"
            df = pd.read_parquet(p)
            cap = SAMPLE_N if n == "s1" else SAMPLE_N * 30
            if len(df) > cap:
                df.iloc[:cap].to_parquet(p, index=False)
    print(f"SAMPLE mode: s1 capped {SAMPLE_N}, pool capped {SAMPLE_N*30}")
for split in ("train", "test"):
    rows = {n: len(pd.read_parquet(f"{WORK}/{split}_{n}.parquet", columns=["entity_id"]))
            for n in ("s1", "s2", "s3")}
    print(split, rows)
for split in ("train", "test"):
    if prep_cached.get(split, False):
        continue
    inputs = [os.path.join(DATA_DIR, split, f"{split}_source{i}.tsv") for i in (1, 2, 3)]
    if not os.path.exists(inputs[0]):
        inputs = [os.path.join(DATA_DIR, f"{split}_source{i}.tsv") for i in (1, 2, 3)]
    prep_cfg = {"split": split, "sample_n": SAMPLE_N,
                "ground_truth_sha256": file_sha256(gt_src),
                "source_bytes": [os.path.getsize(p) for p in inputs]}
    prep_outputs = [f"{WORK}/{split}_{n}.parquet" for n in ("s1", "s2", "s3")] +                    [f"{WORK}/{split}_s1_ids.parquet", f"{WORK}/{split}_pool_ids.parquet"]
    save_slim_ids(split)
    write_files_manifest(os.path.join(WORK, f"_{split}_prep_manifest.json"),
                         f"{split}_prep", prep_cfg, prep_outputs,
                         sum(len(pd.read_parquet(p, columns=["entity_id"])) for p in prep_outputs[:3]))
print("Stage A DONE")


In [ ]:
# S3a — blocking functions (inlined blocking.py)
import math
import json

import os

import shutil

import tempfile

MANIFEST_NAME = '_stage_manifest.json'

def ensure_local_path(path):
    """Normalize an env-derived path and refuse traversal outside its base.

    Stage directories and label paths come from operator-set environment
    variables; this guard keeps a bad value from escaping the intended
    workspace via '..' components."""
    norm = os.path.normpath(os.path.abspath(str(path)))
    if '..' in norm.split(os.sep):
        raise ValueError(f'path escapes its base after normalization: {path}')
    return norm

def files_complete(manifest_path, stage, config, expected_files=None):
    """Validate a manifest for artifacts that live beside or outside it."""
    path = manifest_path
    if not os.path.isfile(path):
        return False
    try:
        with open(path, encoding='utf-8') as f:
            manifest = json.load(f)
        if manifest.get('complete') is not True or manifest.get('stage') != stage or manifest.get('config') != config:
            return False
        files = manifest.get('files', [])
        if not files:
            return False
        for item in files:
            target = item.get('path') or os.path.join(os.path.dirname(path), item['name'])
            if not os.path.isfile(target) or os.path.getsize(target) != item['bytes']:
                return False
        if expected_files is not None and {os.path.abspath(p) for p in expected_files} != {os.path.abspath(item.get('path') or os.path.join(os.path.dirname(path), item['name'])) for item in files}:
            return False
        return True
    except (OSError, ValueError, KeyError, TypeError):
        return False

def is_complete(directory, stage, config):
    """Return true only when config and every recorded output still match."""
    return files_complete(os.path.join(directory, MANIFEST_NAME), stage, config)

def prepare_stage(directory, stage, config):
    """Reuse a verified stage; otherwise clear its own output and start clean."""
    directory = ensure_local_path(directory)
    if is_complete(directory, stage, config):
        return False
    if os.path.isdir(directory):
        shutil.rmtree(directory)
    os.makedirs(directory, exist_ok=True)
    return True

def write_manifest(directory, stage, config, files, total_rows):
    """Atomically mark a stage complete after all outputs have been closed."""
    return write_files_manifest(os.path.join(directory, MANIFEST_NAME), stage, config, files, total_rows)

def write_files_manifest(manifest_path, stage, config, files, total_rows):
    """Atomically record exact output paths, sizes, and row count."""
    directory = os.path.dirname(manifest_path)
    os.makedirs(directory, exist_ok=True)
    items = []
    for path in files:
        if not os.path.isfile(path) or os.path.getsize(path) == 0:
            raise RuntimeError(f'stage output missing or empty: {path}')
        items.append({'name': os.path.basename(path), 'path': os.path.abspath(path), 'bytes': os.path.getsize(path)})
    if not items:
        raise RuntimeError(f'stage {stage} produced no output files')
    manifest = {'stage': stage, 'config': config, 'total_rows': int(total_rows), 'files': items, 'complete': True}
    fd, temp_path = tempfile.mkstemp(prefix='manifest_', suffix='.tmp', dir=directory)
    try:
        with os.fdopen(fd, 'w', encoding='utf-8') as f:
            json.dump(manifest, f, sort_keys=True)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, manifest_path)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)
import sys, os, time, argparse, math

from array import array

import numpy as np

import pandas as pd

def build_index(pool, chunk_rows=1000000):
    """Single-pass index: key -> array('i') of pool row idx. Memory-lean (no big temporaries).

    df caps are applied at QUERY time via make_allowed(); the index stores postings
    for every key so different caps can be tried without rebuilding.
    """
    from collections import defaultdict
    t0 = time.time()
    index = defaultdict(lambda: array('i'))
    dfs = {'c3': {}, 't': {}, 'a': {}, 'sn': {}, 'fn': {}}
    for base in range(0, len(pool), chunk_rows):
        chunk = pool.iloc[base:base + chunk_rows]
        c3_mask = chunk.name_s.str.len() >= 3
        for k, v in ('c3|' + chunk.country[c3_mask] + '|' + chunk.name_s[c3_mask].str[:3]).value_counts().items():
            dfs['c3'][k] = dfs['c3'].get(k, 0) + int(v)
        t = pd.DataFrame({'toks': chunk.name_t.str.split().values, 'c': chunk.country.values}).explode('toks').dropna()
        for k, v in ('t|' + t.c + '|' + t.toks).value_counts().items():
            dfs['t'][k] = dfs['t'].get(k, 0) + int(v)
        a = pd.DataFrame({'toks': chunk.addr_t.str.split().values, 'c': chunk.country.values}).explode('toks').dropna()
        a = a[a.toks.str.len() >= 3]
        for k, v in ('a|' + a.c + '|' + a.toks).value_counts().items():
            dfs['a'][k] = dfs['a'].get(k, 0) + int(v)
        snm = chunk.stno != ''
        for k, v in ('sn|' + chunk.country[snm] + '|' + chunk.stno[snm]).value_counts().items():
            dfs['sn'][k] = dfs['sn'].get(k, 0) + int(v)
        for k, v in ('fn|' + chunk.country + '|' + chunk.name_s).value_counts().items():
            dfs['fn'][k] = dfs['fn'].get(k, 0) + int(v)
        cols = zip(chunk.country.values, chunk.name_s.values, chunk.name_t.values, chunk.addr_t.values, chunk.stno.values)
        for j, (c, ns, nt, at, sn) in enumerate(cols):
            i = base + j
            if len(ns) >= 3:
                index[f'c3|{c}|{ns[:3]}'].append(i)
            for tok in nt.split():
                index[f't|{c}|{tok}'].append(i)
            for tok in at.split():
                if len(tok) >= 3:
                    index[f'a|{c}|{tok}'].append(i)
            if sn:
                index[f'sn|{c}|{sn}'].append(i)
            if ns:
                index[f'fn|{c}|{ns}'].append(i)
        print(f'  chunk {base + len(chunk):,}/{len(pool):,} rows ({time.time() - t0:.0f}s)', flush=True)
    print(f'  index: {len(index):,} keys, {sum((len(v) for v in index.values())):,} postings in {time.time() - t0:.0f}s', flush=True)
    return (index, dfs)

def make_allowed(dfs, token_cap, addr_cap, sn_cap, fn_cap, c3_cap=2000):
    allowed = {'c3': {k for k, v in dfs.get('c3', {}).items() if v <= c3_cap}, 't': {k for k, v in dfs['t'].items() if v <= token_cap}, 'a': {k for k, v in dfs['a'].items() if v <= addr_cap}, 'sn': {k for k, v in dfs['sn'].items() if v <= sn_cap}, 'fn': {k for k, v in dfs['fn'].items() if v <= fn_cap}}
    print(f"  allowed keys: c3={len(allowed['c3'])} t={len(allowed['t'])} a={len(allowed['a'])} sn={len(allowed['sn'])} fn={len(allowed['fn'])}")
    return allowed

def record_keys(country, name_s, name_t, addr_t, stno, allowed):
    out = []
    if len(name_s) >= 3:
        k = f'c3|{country}|{name_s[:3]}'
        if k in allowed.get('c3', ()):
            out.append(k)
    cf = f'{country}|'
    a_t = allowed['t']
    a_a = allowed['a']
    a_sn = allowed['sn']
    a_fn = allowed['fn']
    for t in name_t.split():
        k = f't|{cf}{t}'
        if k in a_t:
            out.append(k)
    for t in addr_t.split():
        if len(t) >= 3:
            k = f'a|{cf}{t}'
            if k in a_a:
                out.append(k)
    if stno:
        k = f'sn|{cf}{stno}'
        if k in a_sn:
            out.append(k)
    k = f'fn|{cf}{name_s}'
    if k in a_fn:
        out.append(k)
    return out

_BLOCK_WEIGHT = {'c3': 0.25, 't': 1.0, 'a': 1.25, 'sn': 1.0, 'fn': 1.5}

def _stable_topk(ids, scores, k):
    """Select descending scores, breaking all cutoff and output ties by pool ID."""
    ids = np.asarray(ids)
    scores = np.asarray(scores)
    if k <= 0 or not len(ids):
        return np.empty(0, dtype=np.int64)
    if len(ids) > k:
        cutoff = np.partition(scores, len(scores) - k)[len(scores) - k]
        above = np.flatnonzero(scores > cutoff)
        ties = np.flatnonzero(scores == cutoff)
        need = k - len(above)
        if need < len(ties):
            tie_order = np.argpartition(ids[ties], need - 1)[:need]
            ties = ties[tie_order]
        selected = np.concatenate((above, ties))
    else:
        selected = np.arange(len(ids))
    order = np.lexsort((ids[selected], -scores[selected]))
    return ids[selected[order]]

def _row_candidates(country, name_s, name_t, addr_t, stno, index, allowed, max_candidates=2000):
    """Rank the union cheaply from weighted block evidence before string scoring."""
    scores = {}
    for key in record_keys(country, name_s, name_t, addr_t, stno, allowed):
        postings = index.get(key)
        if postings is None or not len(postings):
            continue
        family = key.split('|', 1)[0]
        weight = _BLOCK_WEIGHT[family] / math.log2(2 + len(postings))
        for pool_idx in postings:
            scores[pool_idx] = scores.get(pool_idx, 0.0) + weight
    if not scores:
        return np.empty(0, dtype=np.int64)
    ids = np.fromiter(scores.keys(), dtype=np.int64, count=len(scores))
    if max_candidates > 0 and len(ids) > max_candidates:
        vals = np.fromiter(scores.values(), dtype=np.float64, count=len(scores))
        ids = _stable_topk(ids, vals, max_candidates)
    return np.sort(ids)

def candidates_chunk(s1_chunk, base, index, allowed, max_candidates=2000):
    """Return deterministic, evidence-prefiltered pool-idx arrays, one per S1 row."""
    out = []
    cols = zip(s1_chunk.country.values, s1_chunk.name_s.values, s1_chunk.name_t.values, s1_chunk.addr_t.values, s1_chunk.stno.values)
    for country, name_s, name_t, addr_t, stno in cols:
        out.append(_row_candidates(country, name_s, name_t, addr_t, stno, index, allowed, max_candidates))
    return out

def _addr_name_only(p_name, p_addr, query_addrs, cand_addrs):
    """Blank address on either side -> p_addr := p_name (name-only proxy)."""
    both = (np.asarray(cand_addrs) != '') & (np.asarray(query_addrs) != '')
    return np.where(both, p_addr, p_name)

def proxy_scores(s1, pool, cand):
    """Cheap TF-IDF cosine proxy (name, addr) for one S1 row's candidates.
    Name-only where either address is blank — the same rule batch_topk_prune
    applies, so gate2b's recall@K measures the production proxy."""
    if isinstance(s1, pd.Series):
        s1 = s1.to_frame().T
    frame_cos_rows(s1, pool)
    p_name, p_addr = cos_pair_scores(s1, pool, np.arange(len(s1)), cand)
    p_addr = _addr_name_only(p_name, p_addr, s1.addr_n.values[0:1], pool.addr_n.values[cand])
    return (p_name, p_addr)

PROXY_FNS = {'mean': lambda pn, pa: 0.5 * (pn + pa), 'max': np.maximum}

def proxy_topk_indices(s1, pool, cand, k, mode='mean'):
    """Top-k candidate pool indices by the named proxy. Deterministic (stable
    sort). Safe on empty/short inputs. Same proxy gate2b measures recall@K with."""
    if cand.size == 0 or k <= 0:
        return cand[:max(k, 0)] if cand.size else cand
    if cand.size <= k:
        return cand
    cand = np.sort(cand)
    p_name, p_addr = proxy_scores(s1, pool, cand)
    proxy = PROXY_FNS[mode](p_name, p_addr)
    return _stable_topk(cand, proxy, k)

def batch_topk_prune(rows, pool, cands, topk, proxy_mode='mean', pair_budget=100000, rank_all=False):
    """Apply top-K with bounded batched scoring calls and deterministic tie-breaking.

    An oversized single row is scored in slices while retaining only its current
    best K, so no scoring call or temporary score array exceeds pair_budget.
    """
    if pair_budget <= 0:
        raise ValueError('pair_budget must be positive')
    out = [np.sort(c) for c in cands]
    need = [j for j, c in enumerate(out) if c.size > topk or (rank_all and c.size)]
    if not need or topk <= 0:
        return out
    row_names = rows.name_s.values
    row_addrs = rows.addr_n.values
    frame_cos_rows(rows, pool)

    def score(row_pos_rep, candidate_ids):
        p_name, p_addr = cos_pair_scores(rows, pool, row_pos_rep, candidate_ids)
        p_addr = _addr_name_only(p_name, p_addr, row_addrs[row_pos_rep], pool.addr_n.values[candidate_ids])
        return PROXY_FNS[proxy_mode](p_name, p_addr)

    def score_one(j):
        cand = out[j]
        best_ids = np.empty(0, dtype=np.int64)
        best_scores = np.empty(0, dtype=np.float64)
        for base in range(0, len(cand), pair_budget):
            part = cand[base:base + pair_budget]
            n = len(part)
            vals = score(np.full(n, j, dtype=np.int64), part)
            ids = np.concatenate((best_ids, part))
            all_scores = np.concatenate((best_scores, vals))
            best_ids = _stable_topk(ids, all_scores, topk)
            score_by_id = {int(i): float(v) for i, v in zip(ids, all_scores)}
            best_scores = np.fromiter((score_by_id[int(i)] for i in best_ids), dtype=np.float64, count=len(best_ids))
        out[j] = best_ids
    pending = []
    pending_pairs = 0

    def flush():
        nonlocal pending, pending_pairs
        if not pending:
            return
        counts = np.array([len(out[j]) for j in pending], dtype=np.int64)
        ids = np.concatenate([out[j] for j in pending])
        rep = np.repeat(np.asarray(pending, dtype=np.int64), counts)
        vals = score(rep, ids)
        offsets = np.concatenate(([0], np.cumsum(counts)))
        for i, j in enumerate(pending):
            start, end = (offsets[i], offsets[i + 1])
            limit = len(out[j]) if rank_all and len(out[j]) <= topk else topk
            out[j] = _stable_topk(out[j], vals[start:end], limit)
        pending, pending_pairs = ([], 0)
    for j in need:
        size = len(out[j])
        if size > pair_budget:
            flush()
            score_one(j)
            continue
        if pending and pending_pairs + size > pair_budget:
            flush()
        pending.append(j)
        pending_pairs += size
    flush()
    return out

def generate_pairs(s1, index, allowed, pool=None, topk=0, proxy='mean', chunk_rows=2048, pair_budget=1000000, max_candidates=2000):
    """Yield aligned (s1_idx, cand_idx, rank) arrays within a pair budget.

    topk > 0 (with pool passed) truncates each S1 row's candidates to the top-K by
    the selected proxy. Candidate rows are scored in bounded batches.
    """
    for base in range(0, len(s1), chunk_rows):
        chunk = s1.iloc[base:base + chunk_rows]
        cands = candidates_chunk(chunk, base, index, allowed, max_candidates)
        if topk and pool is None:
            raise ValueError('pool is required when topk is enabled')
        if topk:
            cands = batch_topk_prune(chunk, pool, cands, topk, proxy, pair_budget=pair_budget, rank_all=True)
        buf_s1, buf_cand, buf_rank, buf_pairs = ([], [], [], 0)
        for j, c in enumerate(cands):
            if not c.size:
                continue
            if buf_pairs and buf_pairs + len(c) > pair_budget:
                yield (np.concatenate(buf_s1), np.concatenate(buf_cand), np.concatenate(buf_rank))
                buf_s1, buf_cand, buf_rank, buf_pairs = ([], [], [], 0)
            buf_s1.append(np.full(len(c), base + j, dtype=np.int64))
            buf_cand.append(c)
            buf_rank.append(np.arange(len(c), dtype=np.int32))
            buf_pairs += len(c)
        if buf_s1:
            yield (np.concatenate(buf_s1), np.concatenate(buf_cand), np.concatenate(buf_rank))

def load_pool(split, columns=None):
    if columns is not None:
        return pd.concat([pd.read_parquet(f'{WORK}/{split}_s2.parquet', columns=columns), pd.read_parquet(f'{WORK}/{split}_s3.parquet', columns=columns)], ignore_index=True)
    return pd.concat([pd.read_parquet(f'{WORK}/{split}_s2.parquet'), pd.read_parquet(f'{WORK}/{split}_s3.parquet')], ignore_index=True)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--split', default='train')
    ap.add_argument('--token-cap', type=int, default=1000)
    ap.add_argument('--addr-cap', type=int, default=2000)
    ap.add_argument('--sn-cap', type=int, default=2000)
    ap.add_argument('--fn-cap', type=int, default=20000)
    ap.add_argument('--c3-cap', type=int, default=2000)
    ap.add_argument('--max-candidates', type=int, default=2000)
    ap.add_argument('--pair-budget', type=int, default=1000000)
    ap.add_argument('--topk', type=int, default=0, help='per-S1 candidate cap via proxy ranking (0 = off); volume knob chosen by gate2b recall@K')
    ap.add_argument('--proxy', default='mean', choices=sorted(PROXY_FNS), help='proxy aggregation for top-K ranking')
    ap.add_argument('--s1-limit', type=int, default=0)
    ap.add_argument('--out', default='')
    args = ap.parse_args()
    pool = load_pool(args.split)
    s1 = pd.read_parquet(f'{WORK}/{args.split}_s1.parquet')
    if args.s1_limit:
        s1 = s1.iloc[:args.s1_limit]
    print(f'pool={len(pool):,} s1={len(s1):,}')
    index, dfs = build_index(pool)
    allowed = make_allowed(dfs, args.token_cap, args.addr_cap, args.sn_cap, args.fn_cap, args.c3_cap)
    del dfs
    t0 = time.time()
    out_dir = args.out or os.path.join(WORK, f'{args.split}_cands')
    stage_config = {'split': args.split, 'token_cap': args.token_cap, 'addr_cap': args.addr_cap, 'sn_cap': args.sn_cap, 'fn_cap': args.fn_cap, 'c3_cap': args.c3_cap, 'max_candidates': args.max_candidates, 'topk': args.topk, 'proxy': args.proxy, 'pair_budget': args.pair_budget}
    if not prepare_stage(out_dir, f'{args.split}_candidates', stage_config):
        print(f'candidate stage complete and config-matched: {out_dir}')
        return
    total = 0
    output_files = []
    for part, (a, b, rank) in enumerate(generate_pairs(s1, index, allowed, pool=pool, topk=args.topk, proxy=args.proxy, pair_budget=args.pair_budget, max_candidates=args.max_candidates)):
        part_path = os.path.join(out_dir, f'part{part:05d}.parquet')
        pd.DataFrame({'s1_idx': a, 'cand_idx': b, 'cand_rank': rank}).to_parquet(part_path, index=False)
        output_files.append(part_path)
        total += len(a)
        print(f'  part{part:03d}: {len(a):,} pairs (total {total:,}, {time.time() - t0:.0f}s)', flush=True)
    print(f'{total:,} candidate pairs ({total / len(s1):.1f}/S1 avg) -> {out_dir}')
    write_manifest(out_dir, f'{args.split}_candidates', stage_config, output_files, total)

In [ ]:
# S3b — Stage B: recall@K sweep on the val split (K x proxy x caps)
import gc

def make_val_split(save=True):
    s1 = pd.read_parquet(f"{WORK}/train_s1.parquet")
    target_n = min(20_000, len(s1))
    gt_local = pd.read_csv(GT, sep="	", dtype=str, keep_default_na=False)
    degree = gt_local.set_index("source1_entity_id").matched_entity_ids.map(
        lambda x: min(4, len([m for m in x.split(",") if m])))
    labels = s1.entity_id.map(degree).fillna(0).astype(int)
    rng = np.random.default_rng(42)
    chosen = []
    for _, group in s1.assign(_stratum=labels).groupby("_stratum", sort=True):
        ids = group.entity_id.to_numpy()
        take = min(len(ids), int(round(target_n * len(ids) / max(len(s1), 1))))
        chosen.extend(rng.choice(ids, size=take, replace=False).tolist() if take else [])
    if len(chosen) < target_n:
        remaining = s1.loc[~s1.entity_id.isin(chosen), "entity_id"].to_numpy()
        chosen.extend(rng.choice(remaining, size=target_n - len(chosen),
                                 replace=False).tolist())
    chosen = np.asarray(chosen[:target_n], dtype=object)
    rng.shuffle(chosen)
    midpoint = len(chosen) // 2
    calibration_ids = set(chosen[:midpoint])
    tuning_ids = set(chosen[midpoint:])
    assert not calibration_ids.intersection(tuning_ids)
    assert calibration_ids.union(tuning_ids) == set(chosen.tolist())
    if save:
        pd.Series(sorted(calibration_ids)).to_csv(
            f"{WORK}/candidate_calibration_ids.txt", index=False, header=False)
        pd.Series(sorted(tuning_ids)).to_csv(f"{WORK}/val_s1_ids.txt", index=False, header=False)
    return calibration_ids

val_ids = make_val_split()
pool = load_pool("train")
s1 = pd.read_parquet(f"{WORK}/train_s1.parquet")
val = s1[s1.entity_id.isin(val_ids)].reset_index(drop=True)
del s1
gt = pd.read_csv(GT, sep="\t", dtype=str, keep_default_na=False)
tp = gt.assign(m=gt.matched_entity_ids.str.split(",")).explode("m")
tp = tp[tp.m.str.strip() != ""]
tp = tp[tp.source1_entity_id.isin(val_ids)]
del gt
print(f"val S1: {len(val):,} | true pairs: {len(tp):,}", flush=True)

eid2i = {e: i for i, e in enumerate(val.entity_id.values)}          # val row index
pid2i = {e: i for i, e in enumerate(pool.entity_id.values)}
tp_set = set((eid2i[s], pid2i[m]) for s, m in zip(tp.source1_entity_id.values, tp.m.values)
             if s in eid2i and m in pid2i)
tp_by_s1 = {}
for s, m in zip(tp.source1_entity_id.values, tp.m.values):
    if s in eid2i and m in pid2i:
        tp_by_s1.setdefault(eid2i[s], set()).add(pid2i[m])
del tp, pid2i, eid2i
total_true = sum(len(v) for v in tp_by_s1.values())

index, dfs = build_index(pool)
results = []
for setting in _SETTINGS:
    allowed = make_allowed(dfs, *setting)
    hits = {p: {k: 0 for k in KS} for p in PROXY_FNS}
    sizes = {p: {k: 0 for k in KS} for p in PROXY_FNS}
    for base in range(0, len(val), 512):
        chunk = val.iloc[base:base + 512]
        base_cands = candidates_chunk(chunk, base, index, allowed, MAX_CANDIDATES)
        for p in PROXY_FNS:
            ranked = batch_topk_prune(chunk, pool, base_cands, max(KS), p, PAIR_BUDGET,
                                       rank_all=True)
            for j, c in enumerate(ranked):
                truth = tp_by_s1.get(base + j, ())
                for k in KS:
                    top = c[:k]
                    sizes[p][k] += len(top)
                    if truth:
                        hits[p][k] += len(set(top.tolist()) & truth)
        del base_cands, ranked
    for p in PROXY_FNS:
        for k in KS:
            results.append({"caps": str(setting), "proxy": p, "topk": k,
                            "recall": hits[p][k] / total_true if total_true else 0.0,
                            "avg_cands_per_s1": sizes[p][k] / max(len(val), 1),
                            "reduction_vs_naive": sizes[p][k] / max(len(val) * len(pool), 1)})
    del allowed
    gc.collect()
res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))
res_df.to_csv(f"{WORK}/stageB_recall_table.csv", index=False)
if total_true:
    assert res_df.recall.max() > 0.02, "recall ~0: eid2i/positional key bug has recurred"
else:
    print("Stage B WARNING: zero true pairs available in pool for this slice — table is informational only")
best_recall = float(res_df.recall.max())
best_row = res_df.loc[res_df.recall.idxmax()]
cfg_row = res_df[(res_df.proxy == PROXY) & (res_df.topk == TOPK)]
if len(cfg_row):
    chosen = cfg_row.loc[cfg_row.recall.idxmax()]
    print(f"Stage B DONE — CONFIG honored: proxy={PROXY} topk={TOPK} "
          f"recall={chosen.recall:.4%} avg_cands={chosen.avg_cands_per_s1:.1f} "
          f"(table best recall {best_recall:.4%})")
else:
    print(f"Stage B WARNING: CONFIG proxy={PROXY} topk={TOPK} not in sweep (KS={KS}) — "
          f"kept as-is; table best: proxy={best_row.proxy} topk={int(best_row.topk)} "
          f"recall={best_recall:.4%}")
del index, pool, val
gc.collect()

In [ ]:
# S3c — Stage C: train candidate generation with chosen TOPK/PROXY (test-side is
# generated in Section 7 by the SAME functions — no drift)
out_dir = f"{WORK}/train_cands"
stage_c_cfg = {"caps": CAPS, "max_candidates": MAX_CANDIDATES, "topk": TOPK,
               "proxy": PROXY, "pair_budget": PAIR_BUDGET}
if not prepare_stage(out_dir, "train_candidates", stage_c_cfg):
    print("Stage C complete and config-matched")
else:
    pool = load_pool("train")
    s1 = pd.read_parquet(f"{WORK}/train_s1.parquet",
                         columns=["entity_id", "country", "name_s", "name_t",
                                  "addr_n", "addr_t", "stno"])
    index, dfs = build_index(pool)
    allowed = make_allowed(dfs, **CAPS)
    del dfs
    total = 0
    candidate_files = []
    for part, (a, b, rank) in enumerate(generate_pairs(s1, index, allowed, pool=pool,
            topk=TOPK, proxy=PROXY, pair_budget=PAIR_BUDGET,
            max_candidates=MAX_CANDIDATES)):
        part_path = os.path.join(out_dir, f"part{part:05d}.parquet")
        pd.DataFrame({"s1_idx": a, "cand_idx": b, "cand_rank": rank}).to_parquet(part_path, index=False)
        candidate_files.append(part_path)
        total += len(a)
        print(f"  part{part:03d}: {len(a):,} (total {total:,})", flush=True)
    print(f"Stage C DONE: {total:,} train candidate pairs")
    write_manifest(out_dir, "train_candidates", stage_c_cfg, candidate_files, total)
    del index, pool
    gc.collect()

In [ ]:
# S4 — Stage D: pairwise features (inlined features.py)
import sys, os, time, argparse, glob, hashlib

import numpy as np

import pandas as pd

def jaccard(a_tokens, b_tokens):
    if not a_tokens and (not b_tokens):
        return 1.0
    u = len(a_tokens | b_tokens)
    return len(a_tokens & b_tokens) / u if u else 1.0

def load_feats_data(split):
    s1 = pd.read_parquet(f'{WORK}/{split}_s1.parquet')
    pool = pd.concat([pd.read_parquet(f'{WORK}/{split}_s2.parquet'), pd.read_parquet(f'{WORK}/{split}_s3.parquet')], ignore_index=True)
    return (s1, pool)

def _tok_set(v):
    return frozenset(v.split()) if isinstance(v, str) and v else frozenset()

def ensure_entity_feats(frame):
    """Per-entity token sets + scalars, computed once and cached on the frame.

    compute_chunk used to re-split the same entity strings for every 100k-pair
    chunk and take 100k random rows out of multi-million-row arrow-backed
    frames each time - that was the 47 s/part Stage D rate on Kaggle (and the
    memory death). With this cache, feature calculation is pure positional
    lookup + set math. The owner tag guards against attrs propagating through
    iloc (pandas 3.0), same as the cosine cache."""
    cached = frame.attrs.get('_ef')
    if cached is not None and cached.get('owner') == id(frame):
        return cached
    an = frame.addr_n.fillna('').to_numpy(dtype=object)
    cached = {'owner': id(frame), 'raw': frame.business_name.str.lower().map(_tok_set).to_numpy(dtype=object), 'nt': frame.name_t.map(_tok_set).to_numpy(dtype=object), 'nn': frame.name_nt.map(_tok_set).to_numpy(dtype=object), 'at': frame.addr_t.map(_tok_set).to_numpy(dtype=object), 'an': an, 'sn': frame.stno.fillna('').to_numpy(dtype=object), 'len_s': frame.name_s.str.len().to_numpy(dtype=np.float64), 'len_a': frame.addr_n.str.len().to_numpy(dtype=np.float64)}
    frame.attrs['_ef'] = cached
    return cached

def compute_chunk(s1, pool, s1i, ci):
    """All features for one chunk of (s1_idx, cand_idx) arrays."""
    ef1 = ensure_entity_feats(s1)
    efp = ensure_entity_feats(pool)
    e_raw, p_raw = (ef1['raw'], efp['raw'])
    e_nt, p_nt = (ef1['nt'], efp['nt'])
    e_nn, p_nn = (ef1['nn'], efp['nn'])
    e_at, p_at = (ef1['at'], efp['at'])
    e_an, p_an = (ef1['an'], efp['an'])
    n = len(s1i)
    name_jac_raw = np.empty(n)
    name_jac_n = np.empty(n)
    name_jac_s = np.empty(n)
    overlap = np.empty(n, dtype=np.int32)
    addr_jac = np.full(n, np.nan)
    for i in range(n):
        s, c = (s1i[i], ci[i])
        name_jac_raw[i] = jaccard(e_raw[s], p_raw[c])
        name_jac_n[i] = jaccard(e_nn[s], p_nn[c])
        name_jac_s[i] = jaccard(e_nt[s], p_nt[c])
        overlap[i] = len(e_nt[s] & p_nt[c])
        if e_an[s] and p_an[c]:
            addr_jac[i] = jaccard(e_at[s], p_at[c])
    attach_cosine(pool)
    frame_cos_rows(s1, pool)
    name_cos, addr_cos_raw = cos_pair_scores(s1, pool, s1i, ci)
    both_addr = (e_an[s1i] != '') & (p_an[ci] != '')
    addr_cos = np.where(both_addr, addr_cos_raw, np.nan)
    e_sn, p_sn = (ef1['sn'], efp['sn'])
    sn_match = np.full(n, np.nan)
    both = (e_sn[s1i] != '') & (p_sn[ci] != '')
    sn_match[both] = (e_sn[s1i][both] == p_sn[ci][both]).astype(float)
    missing_addr = (p_an[ci] == '').astype(np.int8)
    len_delta_name = np.abs(ef1['len_s'][s1i] - efp['len_s'][ci])
    len_delta_addr = np.abs(ef1['len_a'][s1i] - efp['len_a'][ci])
    out = pd.DataFrame({'s1_idx': s1i, 'cand_idx': ci, 'name_jaccard_raw': name_jac_raw, 'name_jaccard_normalized': name_jac_n, 'name_jaccard_suffix_stripped': name_jac_s, 'name_cos': name_cos, 'addr_jaccard': addr_jac, 'addr_cos': addr_cos, 'token_overlap_count': overlap, 'street_number_match': sn_match, 'missing_address_flag': missing_addr, 'len_delta_name': len_delta_name, 'len_delta_addr': len_delta_addr})
    return out

def label_candidate_pairs(cands, s1_ids, pool_ids, gt_map):
    """Label a bounded candidate block before expensive feature calculation."""
    s1_idx = cands.s1_idx.to_numpy(dtype=np.int64, copy=False)
    cand_idx = cands.cand_idx.to_numpy(dtype=np.int64, copy=False)
    return np.fromiter((pool_ids[ci] in gt_map.get(s1_ids[si], ()) for si, ci in zip(s1_idx, cand_idx)), dtype=np.int8, count=len(cands))

def select_candidate_rows(cands, labels, s1_ids, val_ids, neg_ratio, seed):
    """Keep validation rows, all positives, and sampled train-fold negatives."""
    if not len(cands):
        return (cands, labels)
    s1_idx = cands.s1_idx.to_numpy(dtype=np.int64, copy=False)
    if isinstance(val_ids, (set, frozenset)):
        val_ids = sorted(val_ids)
    val_id_array = np.asarray(val_ids, dtype=object)
    is_val = np.isin(s1_ids[s1_idx], val_id_array)
    rng = np.random.default_rng(seed)
    keep = is_val | (labels == 1) | (rng.random(len(cands)) < neg_ratio)
    return (cands.loc[keep].reset_index(drop=True), labels[keep])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--split', default='train')
    ap.add_argument('--labels', action='store_true', help='join is_match from ground truth (train only)')
    ap.add_argument('--neg-ratio', type=float, default=0.0, help='train-side negative downsample: keep each negative with this probability (0 = keep all). Val-entity rows and positives are always kept so the F_0.5 sweep sees full candidate sets.')
    ap.add_argument('--chunk', type=int, default=100000, help='maximum candidate pairs per feature calculation')
    args = ap.parse_args()
    s1, pool = load_feats_data(args.split)
    cand_files = sorted(glob.glob(os.path.join(WORK, f'{args.split}_cands', 'part*.parquet')))
    if not cand_files:
        raise FileNotFoundError(f'no candidate files under {WORK}/{args.split}_cands/')
    print(f's1={len(s1):,} pool={len(pool):,} candidate_parts={len(cand_files):,}')
    gt_map = None
    val_ids = None
    if args.labels:
        gt = pd.read_csv(GT, sep='\t', dtype=str, keep_default_na=False)
        gt_map = {e: set(m.split(',')) if m else set() for e, m in zip(gt.source1_entity_id, gt.matched_entity_ids)}
        val_path = os.path.join(WORK, 'val_s1_ids.txt')
        if not os.path.exists(val_path):
            raise FileNotFoundError(f'validation IDs are required for sampled labels: {val_path}')
        val_ids = set(pd.read_csv(val_path, header=None)[0].values)
        val_id_array = np.asarray(sorted(val_ids), dtype=object)
        with open(GT, 'rb') as f:
            gt_hash = hashlib.file_digest(f, 'sha256').hexdigest()
    else:
        gt_hash = ''
    feat_dir = f'{WORK}/feats_{args.split}'
    stage_config = {'split': args.split, 'labels': args.labels, 'neg_ratio': args.neg_ratio, 'chunk_pairs': args.chunk, 'validation_ids': sorted(val_ids) if val_ids is not None else [], 'ground_truth_sha256': gt_hash, 'candidate_manifest_bytes': os.path.getsize(os.path.join(WORK, f'{args.split}_cands', '_stage_manifest.json'))}
    if not prepare_stage(feat_dir, f'features_{args.split}', stage_config):
        print(f'feature stage complete and config-matched: {feat_dir}')
        return
    s1_ids = s1.entity_id.values
    pool_ids = pool.entity_id.values
    t00 = time.time()
    part = 0
    pair_total = 0
    feature_total = 0
    output_files = []
    for cand_file in cand_files:
        cand_part = pd.read_parquet(cand_file)
        pair_total += len(cand_part)
        for base in range(0, len(cand_part), args.chunk):
            t0 = time.time()
            blk = cand_part.iloc[base:base + args.chunk]
            labels = None
            if gt_map is not None:
                labels = label_candidate_pairs(blk, s1_ids, pool_ids, gt_map)
                if args.neg_ratio > 0:
                    blk, labels = select_candidate_rows(blk, labels, s1_ids, val_id_array, args.neg_ratio, seed=42 + part)
            if len(blk):
                df = compute_chunk(s1, pool, blk.s1_idx.values, blk.cand_idx.values)
                if labels is not None:
                    df['is_match'] = labels
                if 'cand_rank' in blk:
                    df['cand_rank'] = blk.cand_rank.to_numpy(dtype=np.int32, copy=False)
                output_path = os.path.join(feat_dir, f'part{part:06d}.parquet')
                df.to_parquet(output_path, index=False)
                output_files.append(output_path)
                feature_total += len(df)
                pos = float(df.is_match.mean()) if labels is not None else float('nan')
                print(f'  part{part:06d}: {len(df):,} rows ({time.time() - t0:.0f}s, pos_rate={pos:.4f})', flush=True)
            part += 1
        del cand_part
    print(f'done in {time.time() - t00:.0f}s -> {feat_dir}/; read {pair_total:,} candidate pairs in bounded chunks')
    write_manifest(feat_dir, f'features_{args.split}', stage_config, output_files, feature_total)
import glob as _glob
cand_files = sorted(_glob.glob(f"{WORK}/train_cands/part*.parquet"))
s1 = pd.read_parquet(f"{WORK}/train_s1.parquet")
pool = load_pool("train")
gt = pd.read_csv(GT, sep="\t", dtype=str, keep_default_na=False)
gt_map = {e: (set(m.split(",")) if m else set())
          for e, m in zip(gt.source1_entity_id, gt.matched_entity_ids)}
val_ids = set(pd.read_csv(f"{WORK}/val_s1_ids.txt", header=None)[0].values)
val_id_array = np.asarray(sorted(val_ids), dtype=object)
feat_dir = f"{WORK}/feats_train"
feat_cfg = {"neg_ratio": NEG_RATIO, "seed": 42, "chunk_pairs": 100_000,
            "candidate_config": stage_c_cfg,
            "candidate_manifest_bytes": os.path.getsize(f"{WORK}/train_cands/_stage_manifest.json"),
            "validation_ids": sorted(val_ids),
            "ground_truth_sha256": file_sha256(gt_src)}
run_stage_d = prepare_stage(feat_dir, "train_features", feat_cfg)
if not run_stage_d:
    print("Stage D complete and config-matched")
else:
    feat_files = []
    feature_total = 0
    s1_ids = s1.entity_id.values
    pool_ids = pool.entity_id.values
    t00 = time.time()
    part = 0
    for cand_file in cand_files:
        cand_part = pd.read_parquet(cand_file)
        for base in range(0, len(cand_part), 100_000):
            t0 = time.time()
            blk = cand_part.iloc[base:base + 100_000]
            labels = label_candidate_pairs(blk, s1_ids, pool_ids, gt_map)
            blk, labels = select_candidate_rows(blk, labels, s1_ids, val_id_array,
                                                NEG_RATIO, seed=42 + part)
            if len(blk):
                df = compute_chunk(s1, pool, blk.s1_idx.values, blk.cand_idx.values)
                df["is_match"] = labels
                if "cand_rank" in blk:
                    df["cand_rank"] = blk.cand_rank.to_numpy(dtype=np.int32)
                out_path = f"{feat_dir}/part{part:06d}.parquet"
                df.to_parquet(out_path, index=False)
                feat_files.append(out_path); feature_total += len(df)
                print(f"  part{part:06d}: {len(df):,} rows ({time.time()-t0:.0f}s)", flush=True)
            part += 1
        del cand_part
    write_manifest(feat_dir, "train_features", feat_cfg, feat_files, feature_total)
    print(f"Stage D DONE in {time.time()-t00:.0f}s; sampled before feature calculation")

In [ ]:
# S5 — Stage E: adversarial identical-name gate (must PASS)
files = sorted(glob.glob(f"{WORK}/feats_train/part*.parquet"))
adv_parts, pos_parts = [], []
for f in files:
    df_part = pd.read_parquet(f, columns=["name_jaccard_suffix_stripped", "addr_jaccard", "is_match"])
    adv_parts.append(df_part.loc[(df_part.name_jaccard_suffix_stripped >= 0.99) &
                                 (df_part.is_match == 0), "addr_jaccard"].dropna().to_numpy())
    pos_parts.append(df_part.loc[(df_part.name_jaccard_suffix_stripped >= 0.99) &
                                 (df_part.is_match == 1), "addr_jaccard"].dropna().to_numpy())
    del df_part
adv = np.concatenate(adv_parts) if adv_parts else np.empty(0)
adv_pos = np.concatenate(pos_parts) if pos_parts else np.empty(0)
if not len(adv) or not len(adv_pos):
    print("Stage E SKIP: no identical-name pairs in this slice (tiny sample mode)")
else:
    gap = float(np.median(adv_pos) - np.median(adv))
    print(f"adversarial subset: {len(adv):,} neg / {len(adv_pos):,} pos")
    print(f"addr_jaccard medians: pos={np.median(adv_pos):.4f} "
          f"adv_neg={np.median(adv):.4f}")
    GATE_E_PASS = gap > 0.1
    assert GATE_E_PASS, 'address features do not separate identical-name negatives - halting pipeline'
    print(f"Stage E {'PASS' if GATE_E_PASS else 'FAIL'}: median addr_jaccard gap = {gap:.4f} "
          f"(threshold 0.1)")
print("Stage E complete")
# S5 gate complete


In [ ]:
# S6a — macro F_0.5 evaluator (inlined evaluate.py) + assertions
import numpy as np

def f05_entity(pred, true):
    p = set(pred)
    t = set(true)
    if not p and (not t):
        return 1.0
    inter = len(p & t)
    prec = 1.0 if not p else inter / len(p)
    rec = 1.0 if not t else inter / len(t)
    den = 0.25 * prec + rec
    return 1.25 * prec * rec / den if den else 0.0

def macro_f05(preds_by_entity, truths_by_entity):
    """preds/truths: dict entity_id -> iterable of matched ids (or lists aligned by key)."""
    pred_keys, truth_keys = (set(preds_by_entity), set(truths_by_entity))
    if pred_keys != truth_keys:
        raise ValueError('prediction and truth entity populations differ')
    keys = pred_keys
    scores = [f05_entity(preds_by_entity.get(k, ()), truths_by_entity.get(k, ())) for k in keys]
    return float(np.mean(scores))

def bucket_report(preds_by_entity, truths_by_entity, thresholds=None):
    """Macro F_0.5 broken out by true match-count bucket: 0 / 1 / 2-3 / 4+."""
    buckets = {'0': [], '1': [], '2-3': [], '4+': []}
    pred_keys, truth_keys = (set(preds_by_entity), set(truths_by_entity))
    if pred_keys != truth_keys:
        raise ValueError('prediction and truth entity populations differ')
    keys = pred_keys
    for k in keys:
        n = len(truths_by_entity.get(k, ()))
        b = '0' if n == 0 else '1' if n == 1 else '2-3' if n <= 3 else '4+'
        buckets[b].append(f05_entity(preds_by_entity.get(k, ()), truths_by_entity.get(k, ())))
    return {b: float(np.mean(v)) if v else float('nan') for b, v in buckets.items()}

def demo():
    assert abs(f05_entity(['a', 'b', 'd'], ['a', 'b', 'c']) - 2 / 3) < 1e-12
    assert f05_entity([], []) == 1.0
    assert f05_entity(['x'], []) == 0.0
    assert f05_entity([], ['x']) == 0.0
    m = macro_f05({'e1': ['a', 'b', 'd'], 'e2': []}, {'e1': ['a', 'b', 'c'], 'e2': []})
    assert abs(m - (2 / 3 + 1.0) / 2) < 1e-12
    print('evaluate.py demo: all F_0.5 assertions passed')

assert abs(f05_entity(["a", "b", "d"], ["a", "b", "c"]) - 2 / 3) < 1e-12
assert f05_entity([], []) == 1.0 and f05_entity(["x"], []) == 0.0 and f05_entity([], ["x"]) == 0.0
m = macro_f05({"e1": ["a", "b", "d"], "e2": []}, {"e1": ["a", "b", "c"], "e2": []})
assert abs(m - (2 / 3 + 1.0) / 2) < 1e-12
print("F_0.5 assertions passed")

In [ ]:
# S6b-pre — model config contract (inlined train.py)
import sys, os, glob, argparse, time, json

import numpy as np

import pandas as pd

import lightgbm as lgb

def sweep_cutoffs_thresholds(proba, entity_index, candidate_rank, labels, true_counts, cutoffs, thresholds):
    """Vectorized macro F0.5 sweep over top-K prefixes and score thresholds."""
    proba = np.asarray(proba, dtype=np.float64)
    entity_index = np.asarray(entity_index, dtype=np.int64)
    candidate_rank = np.asarray(candidate_rank, dtype=np.int32)
    labels = np.asarray(labels, dtype=np.int8)
    true_counts = np.asarray(true_counts, dtype=np.int64)
    if not len(proba) == len(entity_index) == len(candidate_rank) == len(labels):
        raise ValueError('validation arrays must have the same number of pairs')
    n_entities = len(true_counts)
    if np.any(entity_index < 0) or np.any(entity_index >= n_entities):
        raise ValueError('entity_index must address the declared validation population')
    total_true = int(true_counts.sum())
    rows = []
    for k in cutoffs:
        in_topk = candidate_rank < k
        n_cand = np.bincount(entity_index[in_topk], minlength=n_entities)
        candidate_recall = int(labels[in_topk].sum()) / total_true if total_true else 1.0
        for threshold in thresholds:
            selected = in_topk & (proba >= threshold)
            predicted = np.bincount(entity_index[selected], minlength=n_entities)
            true_positive = np.bincount(entity_index[selected], weights=labels[selected], minlength=n_entities)
            precision = np.divide(true_positive, predicted, out=np.ones(n_entities, dtype=np.float64), where=predicted > 0)
            recall = np.divide(true_positive, true_counts, out=np.ones(n_entities, dtype=np.float64), where=true_counts > 0)
            denominator = 0.25 * precision + recall
            scores = np.divide(1.25 * precision * recall, denominator, out=np.zeros(n_entities, dtype=np.float64), where=denominator > 0)
            rows.append({'topk': int(k), 'threshold': float(threshold), 'macro_f05': float(scores.mean()) if n_entities else float('nan'), 'avg_candidates_per_s1': float(n_cand.mean()) if n_entities else 0.0, 'candidate_recall': candidate_recall})
    return pd.DataFrame(rows)

def choose_best_operating_point(table, tolerance=0.005):
    """Maximize macro F0.5; near-ties prefer fewer candidates, then recall."""
    if table.empty or not np.isfinite(table.macro_f05).any():
        raise ValueError('operating-point table has no finite F0.5 scores')
    best_score = float(table.macro_f05.max())
    eligible = table[table.macro_f05 >= best_score - tolerance]
    chosen = eligible.sort_values(['avg_candidates_per_s1', 'candidate_recall', 'macro_f05', 'topk'], ascending=[True, False, False, True], kind='stable').iloc[0]
    return chosen

def make_model_config(threshold, val_macro_f05, per_bucket_f05):
    """The exact Kaggle-return contract consumed by infer.py."""
    return {'threshold': float(threshold), 'val_macro_f05': float(val_macro_f05), 'per_bucket_f05': {k: float(v) for k, v in per_bucket_f05.items()}}

In [ ]:
# S6b — Stage F: LightGBM (ES on train-fold slice) + F_0.5 threshold sweep
train_cfg = {"candidate_config": stage_c_cfg, "topk": TOPK, "proxy": PROXY,
             "caps": CAPS, "neg_ratio": NEG_RATIO,
             "ground_truth_sha256": file_sha256(gt_src),
             "feature_manifest_bytes": os.path.getsize(f"{WORK}/feats_train/_stage_manifest.json"),
             "tuning_ids": sorted(val_ids)}
train_manifest = os.path.join(WORK, "_train_model_manifest.json")
model_outputs = [f"{WORK}/model.txt", f"{WORK}/model_config.json"]
if files_complete(train_manifest, "train_model", train_cfg, model_outputs):
    print("Stage F complete and config-matched")
else:
    for stale_model in model_outputs:
        if os.path.exists(stale_model): os.remove(stale_model)
    feature_files = sorted(glob.glob(f"{WORK}/feats_train/part*.parquet"))
    assert feature_files, "no feature files — Stage D did not produce training data"
    with open(f"{WORK}/feats_train/_stage_manifest.json", encoding="utf-8") as mf:
        n_features = int(json.load(mf)["total_rows"])
    assert n_features > 0, "feature manifest reports zero candidate rows"
    X = np.memmap(f"{WORK}/_train_X.float32", mode="w+", dtype=np.float32,
                  shape=(n_features, len(FEATURES)))
    y = np.memmap(f"{WORK}/_train_y.int8", mode="w+", dtype=np.int8, shape=(n_features,))
    s1_index = np.memmap(f"{WORK}/_train_s1.int32", mode="w+", dtype=np.int32, shape=(n_features,))
    candidate_index = np.memmap(f"{WORK}/_train_cand.int32", mode="w+", dtype=np.int32, shape=(n_features,))
    candidate_rank = np.memmap(f"{WORK}/_train_rank.int32", mode="w+", dtype=np.int32, shape=(n_features,))
    cursor = 0
    for feature_file in feature_files:
        part_df = pd.read_parquet(feature_file)
        end = cursor + len(part_df)
        X[cursor:end] = part_df[FEATURES].to_numpy(dtype=np.float32)
        y[cursor:end] = part_df.is_match.to_numpy(dtype=np.int8)
        s1_index[cursor:end] = part_df.s1_idx.to_numpy(dtype=np.int32)
        candidate_index[cursor:end] = part_df.cand_idx.to_numpy(dtype=np.int32)
        candidate_rank[cursor:end] = part_df.cand_rank.to_numpy(dtype=np.int32)
        cursor = end
        del part_df
    for mapped in (X, y, s1_index, candidate_index, candidate_rank): mapped.flush()
    s1_ids = pd.read_parquet(f"{WORK}/train_s1_ids.parquet")["entity_id"].values
    pool_ids = pd.read_parquet(f"{WORK}/train_pool_ids.parquet")["entity_id"].values
    val_ids_arr = pd.read_csv(f"{WORK}/val_s1_ids.txt", header=None)[0].values
    id_to_s1 = {e: i for i, e in enumerate(s1_ids)}
    val_s1_idx = np.asarray([id_to_s1[e] for e in val_ids_arr], dtype=np.int32)
    val_mask = np.isin(s1_index, val_s1_idx)
    gt_all = pd.read_csv(GT, sep="\t", dtype=str, keep_default_na=False)
    truth_by = {e: (set(m.split(",")) if m else set())
                for e, m in zip(gt_all.source1_entity_id, gt_all.matched_entity_ids)}
    tr_mask = ~val_mask
    train_only = np.unique(s1_index[tr_mask])
    es_ids = np.random.default_rng(1).choice(
        train_only, size=max(1, len(train_only) // 10), replace=False)
    es_mask = np.isin(s1_index, es_ids) & tr_mask
    fit_mask = tr_mask & ~es_mask
    Xtr, ytr = X[fit_mask], y[fit_mask]
    print(f"fit: {len(Xtr):,} (pos {float(ytr.mean()):.4%}) | es: {int(es_mask.sum()):,}")
    params = {"objective": "binary", "metric": "average_precision", "learning_rate": 0.08,
              "num_leaves": 128, "min_data_in_leaf": 200, "feature_fraction": 0.9,
              "bagging_fraction": 0.9, "bagging_freq": 1, "num_threads": -1,
              "scale_pos_weight": float((1 - ytr.mean()) / max(ytr.mean(), 1e-9))}
    dtr = lgb.Dataset(Xtr, ytr)
    dval = lgb.Dataset(X[es_mask], y[es_mask], reference=dtr)
    model = lgb.train(params, dtr, valid_sets=[dval], num_boost_round=2000,
                      callbacks=[lgb.early_stopping(100)])
    v_p = model.predict(X[val_mask])
    v_s1 = s1_ids[s1_index[val_mask]]
    v_c = pool_ids[candidate_index[val_mask]]
    tune_ids = list(dict.fromkeys(val_ids_arr.tolist()))
    assert len(tune_ids) == len(val_ids_arr), "tuning entity IDs must be unique"
    eid = {e: i for i, e in enumerate(tune_ids)}
    entity_index = np.fromiter((eid[e] for e in v_s1), dtype=np.int32, count=len(v_s1))
    ranks = candidate_rank[val_mask]
    labels = y[val_mask]
    pool_id_set = set(pool_ids.tolist())
    true_counts = np.asarray([len(truth_by.get(e, set()) & pool_id_set) for e in tune_ids], dtype=np.int32)
    operating = sweep_cutoffs_thresholds(v_p, entity_index, ranks, labels, true_counts,
        cutoffs=sorted(set([100, 200, 300, 500, TOPK])),
        thresholds=np.round(np.arange(0.10, 0.951, 0.02), 3))
    sw = operating
    print(sw.to_string(index=False))
    best = choose_best_operating_point(sw, tolerance=0.005)
    pred = {e: set() for e in tune_ids}
    sel = (v_p >= best.threshold) & (ranks < int(best.topk))
    for e, m_ in zip(v_s1[sel], v_c[sel]): pred[e].add(m_)
    truth_tune = {e: truth_by.get(e, ()) for e in tune_ids}
    buckets = bucket_report(pred, truth_tune)
    print(f"BEST topk={best.topk} threshold={best.threshold} val_macro_f05={best.macro_f05:.4f} "
          f"avg_candidates={best.avg_candidates_per_s1:.1f} candidate_recall={best.candidate_recall:.4%}")
    print("per-bucket F_0.5:", buckets)
    model.save_model(f"{WORK}/model.txt")
    model_cfg = make_model_config(best.threshold, best.macro_f05, buckets)
    model_cfg["topk"] = int(best.topk)
    model_cfg["proxy"] = PROXY
    model_cfg["caps"] = CAPS
    model_cfg["max_candidates"] = MAX_CANDIDATES
    model_cfg["candidate_recall"] = float(best.candidate_recall)
    model_cfg["avg_candidates_per_s1"] = float(best.avg_candidates_per_s1)
    pd.Series(model_cfg) \
        .to_json(f"{WORK}/model_config.json")
    write_files_manifest(train_manifest, "train_model", train_cfg, model_outputs, n_features)
    print("Stage F DONE")

In [ ]:
# S7 — Stage G: test inference + inline validator (inlined infer.py)
import sys, os, time, argparse

import numpy as np

import pandas as pd

def group_rows(s1_idx_arr, cand_id_arr, s1_ids, keep_mask=None):
    """Per-entity id lists for one chunk. Returns (all_ids_by_entity, kept_ids_by_entity)."""
    all_by = {}
    kept_by = {}
    if keep_mask is not None:
        for s, c, k in zip(s1_idx_arr, cand_id_arr, keep_mask):
            e = s1_ids[s]
            all_by.setdefault(e, []).append(c)
            if k:
                kept_by.setdefault(e, []).append(c)
    else:
        for s, c in zip(s1_idx_arr, cand_id_arr):
            all_by.setdefault(s1_ids[s], []).append(c)
    return (all_by, kept_by)

test_pool = load_pool("test")
t1 = pd.read_parquet(f"{WORK}/test_s1.parquet")
index_t, dfs_t = build_index(test_pool)
allowed_t = make_allowed(dfs_t, **CAPS)
del dfs_t
model = lgb.Booster(model_file=f"{WORK}/model.txt")
cfg = pd.read_json(f"{WORK}/model_config.json", typ="series").to_dict()
threshold = float(cfg["threshold"])
TOPK = int(cfg.get("topk", TOPK))
PROXY = str(cfg.get("proxy", PROXY))
CAPS = dict(cfg.get("caps", CAPS))
MAX_CANDIDATES = int(cfg.get("max_candidates", MAX_CANDIDATES))
s1_ids_t = t1.entity_id.values
pool_ids_t = test_pool.entity_id.values
m_path = os.path.join(OUTPUT_DIR, "matching_results.tsv")
c_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
inference_cfg = {"caps": CAPS, "max_candidates": MAX_CANDIDATES, "topk": TOPK,
                 "proxy": PROXY, "pair_budget": PAIR_BUDGET,
                 "threshold": threshold, "model_bytes": os.path.getsize(f"{WORK}/model.txt")}
prepare_stage(OUTPUT_DIR, "test_inference", inference_cfg)
first = True
n_cand = n_kept = 0
t0 = time.time()
W = 2_048
for base in range(0, len(t1), W):
    chunk = t1.iloc[base:base + W]
    chunk_ids = chunk.entity_id.values
    cands = candidates_chunk(chunk, base, index_t, allowed_t, MAX_CANDIDATES)
    if TOPK:
        cands = batch_topk_prune(chunk, test_pool, cands, TOPK, PROXY, PAIR_BUDGET)
    counts = [len(c) for c in cands]
    si = np.repeat(np.arange(base, base + len(chunk), dtype=np.int64), counts)
    ci = np.concatenate(cands) if cands else np.empty(0, dtype=np.int64)
    del cands
    cand_by, kept_by = {}, {}
    if len(si):
        offsets = np.concatenate(([0], np.cumsum(counts)))
        for j, e in enumerate(chunk_ids):
            cand_by[e] = pool_ids_t[ci[offsets[j]:offsets[j + 1]]].tolist()
        for pair_base in range(0, len(si), PAIR_BUDGET):
            pair_end = min(pair_base + PAIR_BUDGET, len(si))
            si_part, ci_part = si[pair_base:pair_end], ci[pair_base:pair_end]
            df = compute_chunk(t1, test_pool, si_part, ci_part)
            keep = model.predict(df[FEATURES]) >= threshold
            _, kept_part = group_rows(si_part, pool_ids_t[ci_part], s1_ids_t, keep)
            for e, matches in kept_part.items():
                kept_by.setdefault(e, []).extend(matches)
            del df, si_part, ci_part, keep, kept_part
        for e, ms in kept_by.items():
            assert set(ms).issubset(set(cand_by.get(e, ()))), f"match outside candidates: {e}"
    hdr = first
    pd.DataFrame([(e, ",".join(dict.fromkeys(kept_by.get(e, ())))) for e in chunk_ids],
                 columns=["source1_entity_id", "matched_entity_ids"]) \
        .to_csv(m_path, sep="\t", index=False, header=hdr, mode="w" if first else "a",
                encoding="utf-8", lineterminator="\n")
    pd.DataFrame([(e, ",".join(dict.fromkeys(cand_by.get(e, ())))) for e in chunk_ids],
                 columns=["source1_entity_id", "candidate_entity_ids"]) \
        .to_csv(c_path, sep="\t", index=False, header=hdr, mode="w" if first else "a",
                encoding="utf-8", lineterminator="\n")
    first = False
    n_cand += len(si); n_kept += sum(len(v) for v in kept_by.values())
    print(f"  {base + len(chunk):,}/{len(t1):,}: {len(si):,} pairs, {n_kept:,} kept "
          f"({time.time()-t0:.0f}s)", flush=True)
    del si, ci, cand_by, kept_by
del index_t, test_pool
gc.collect()
print(f"Stage G inference DONE: {len(t1):,} rows, {n_cand:,} pairs, {n_kept:,} kept")

In [ ]:
# S7b — inline submission validator (reimplements utils/validate_submission.py checks)
def validate_submission(matching_path, candidate_path, test_dir):
    required = set()
    source1_path = os.path.join(test_dir, "test", "test_source1.tsv")
    if not os.path.exists(source1_path):
        source1_path = os.path.join(test_dir, "test_source1.tsv")
    with open(source1_path, encoding="utf-8") as f:
        next(f)
        required = {l.split("\t", 1)[0].strip() for l in f if l.strip()}
    seen, dup_rows, intra, self_m, wrong_pfx = set(), set(), set(), set(), set()
    n_rows = empties = 0
    errs = []
    with open(matching_path, encoding="utf-8") as f, open(candidate_path, encoding="utf-8") as cf:
        header = f.readline().rstrip("\n").split("\t")
        cheader = cf.readline().rstrip("\n").split("\t")
        assert header == ["source1_entity_id", "matched_entity_ids"], header
        assert cheader == ["source1_entity_id", "candidate_entity_ids"], cheader
        for line in f:
            s1, tab, rest = line.partition("\t")
            if not tab:
                continue
            n_rows += 1
            candidate_line = cf.readline()
            assert candidate_line, f"candidate file ended before matching row {s1}"
            cand_s1, ctab, cand_rest = candidate_line.rstrip("\n").partition("\t")
            assert ctab and cand_s1 == s1, f"candidate/matching row mismatch for {s1}"
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)
            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            cand_ids = cand_rest.rstrip("\n").split(",") if cand_rest.strip() else []
            if not set(ids).issubset(set(cand_ids)):
                errs.append(f"matches outside candidate list for {s1}")
            if len(ids) != len(set(ids)):
                intra.add(s1)
            for mid in set(ids):
                if mid.startswith("S1-"):
                    self_m.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_pfx.add(mid)
            if not ids:
                empties += 1
        if cf.readline():
            errs.append("candidate file has extra rows")
    if required - seen:
        errs.append(f"{len(required - seen)} required S1 entities missing")
    if seen - required:
        errs.append(f"{len(seen - required)} unknown S1 rows")
    if dup_rows:
        errs.append(f"{len(dup_rows)} duplicate rows")
    if intra:
        errs.append(f"{len(intra)} lists with duplicate ids")
    if self_m:
        errs.append(f"{len(self_m)} self-matches")
    if wrong_pfx:
        errs.append(f"{len(wrong_pfx)} wrong-prefix ids")
    print(f"  matching_results.tsv: {n_rows} rows ({empties} empty)")
    if errs:
        print("FAIL:", "; ".join(errs))
    else:
        print("PASS — no blocking issues found.")
    return not errs

ok = validate_submission(m_path, c_path, DATA_DIR)
assert ok, 'submission validation failed'
write_manifest(OUTPUT_DIR, "test_inference", inference_cfg,
               [m_path, c_path], len(t1))


In [ ]:
# S8 — Stage H: bundle kaggle_return.zip
out = "/kaggle/working/kaggle_return.zip" if os.path.isdir("/kaggle/working") \
    else os.path.join(WORK, "kaggle_return.zip")
if os.path.exists(out):
    os.remove(out)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUTPUT_DIR):
        if f == MANIFEST_NAME:
            continue
        zf.write(os.path.join(OUTPUT_DIR, f), "output/" + f)
    for f in ("model.txt", "model_config.json"):
        p = os.path.join(WORK, f)
        if os.path.exists(p):
            zf.write(p, f)
print(f"kaggle_return.zip: {os.path.getsize(out)/1e6:.1f} MB")
print("DONE — fill Documentation_template.md with the Stage B table + Stage F numbers")